# Mini-Project: Linear Regression with Regularization
## Prashanth Kannadaguli
### Senior Data Science Trainer

## Problem Statement

Predict the bike-sharing counts per hour based on the features including weather, day, time, humidity, wind speed, season e.t.c.

## Learning Objectives

At the end of the mini-project, you will be able to :

* perform data exploration and visualization
* implement linear regression using sklearn and optimization
* apply regularization on regression using Lasso, Ridge and Elasticnet techniques
* calculate and compare the MSE value of each regression technique
* analyze the features that are best contributing to the target

### Dataset

The dataset chosen for this mini-project is [Bike Sharing Dataset](https://archive.ics.uci.edu/ml/datasets/bike+sharing+dataset).  This dataset contains the hourly and daily count of rental bikes between the years 2011 and 2012 in the capital bike share system with the corresponding weather and seasonal information. This dataset consists of 17389 instances of 16 features.

Bike sharing systems are a new generation of traditional bike rentals where the whole process from membership, rental and return has become automatic. Through these systems, the user can easily rent a bike from a particular position and return to another position. Currently, there are about over 500 bike-sharing programs around the world which is composed of over 500 thousand bicycles. Today, there exists great interest in these systems due to their important role in traffic, environmental and health issues.

Apart from interesting real world applications of bike sharing systems, the characteristics of data being generated by these systems make them attractive for the research. As opposed to other transport services such as bus or subway, the duration of travel, departure and arrival position are explicitly recorded in these systems. This feature turns bike sharing system into a virtual sensor network that can be used for sensing mobility in the city. Hence, it is expected that the most important events in the city could be detected via monitoring these data.

<img src="https://s26551.pcdn.co/wp-content/uploads/2012/02/resize-va-sq-bikeshare.jpg" alt="drawing" width="400"/>

### Data Fields

* dteday - hourly date
* season - 1 = spring, 2 = summer, 3 = fall, 4 = winter
* hr - hour
* holiday - whether the day is considered a holiday
* workingday - whether the day is neither a weekend nor holiday
* weathersit -<br>
    1 - Clear, Few clouds, Partly cloudy, Partly cloudy <br>
    2 - Mist + Cloudy, Mist + Broken clouds, Mist + Few clouds, Mist<br>
    3 - Light Snow, Light Rain + Thunderstorm + Scattered clouds, Light Rain + Scattered clouds<br>
    4 - Heavy Rain + Ice Pallets + Thunderstorm + Mist, Snow + Fog<br>   
* temp - temperature in Celsius
* atemp - "feels like" temperature in Celsius
* humidity - relative humidity
* windspeed - wind speed
* casual - number of non-registered user rentals initiated
* registered - number of registered user rentals initiated
* cnt - number of total rentals

## Information

**Regularization:** It is a form of regression that shrinks the coefficient estimates towards zero. In other words, this technique discourages learning a more complex or flexible model, to avoid the risk of overfitting. A simple relation for linear regression looks like this.

$Y ≈ β_0 + β_1 X_1 + β_2 X_2 + …+ β_p X_p$

 Here $Y$ represents the learned relation and $β$ represents the coefficient estimates for different variables or predictors(X).

 If there is noise in the training data, then the estimated coefficients won’t generalize well to the future data. This is where regularization comes in and shrinks or regularizes these learned estimates towards zero.

Below are the Regularization techniques:

 * Ridge Regression
 * Lasso Regression
 * Elasticnet Regression

#### Importing Necessary Packages

In [ ]:
# Loading the Required Packages
import pandas as pd
import numpy as np
from sklearn import linear_model
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score

In [ ]:
# Build hour.csv from the lineardata.pdf (run this once)
from pypdf import PdfReader
import os

if not os.path.exists('hour.csv'):
    print("Building hour.csv from lineardata.pdf...")
    r = PdfReader('lineardata.pdf')
    col1_rows, col2_rows, col3_rows = [], [], []
    
    for pg_idx in range(len(r.pages)):
        text = r.pages[pg_idx].extract_text()
        if not text: continue
        lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
        if not lines: continue
        page_mod = pg_idx % 3
        start = 1 if pg_idx < 3 else 0
        for line in lines[start:]:
            parts = line.split()
            if page_mod == 0:
                if len(parts) >= 6 and parts[0].isdigit():
                    col1_rows.append(parts[:6])
            elif page_mod == 1:
                if len(parts) >= 6:
                    try: float(parts[4]); col2_rows.append(parts[:6])
                    except: pass
            else:
                if len(parts) >= 5:
                    try: float(parts[0]); int(parts[2]); col3_rows.append(parts[:5])
                    except: pass
    
    min_len = min(len(col1_rows), len(col2_rows), len(col3_rows))
    df1 = pd.DataFrame(col1_rows[:min_len], columns=['instant','dteday','season','yr','mnth','hr'])
    df2 = pd.DataFrame(col2_rows[:min_len], columns=['holiday','weekday','workingday','weathersit','temp','atemp'])
    df3 = pd.DataFrame(col3_rows[:min_len], columns=['hum','windspeed','casual','registered','cnt'])
    raw = pd.concat([df1, df2, df3], axis=1)
    for col in ['instant','season','yr','mnth','hr','holiday','weekday','workingday','weathersit','casual','registered','cnt']:
        raw[col] = raw[col].astype(int)
    for col in ['temp','atemp','hum','windspeed']:
        raw[col] = raw[col].astype(float)
    raw.to_csv('hour.csv', index=False)
    print(f"Saved hour.csv with shape {raw.shape}")
else:
    print("hour.csv already exists, skipping build.")


### Data Loading

In [ ]:
# Read the hour.csv file
df = pd.read_csv('hour.csv')
print("Dataset shape:", df.shape)


print the first five rows of dataset

In [ ]:
# print the first five rows of dataset
df.head()


print the datatypes of the columns

In [ ]:
# print the datatypes of the columns
df.dtypes


### Task flow with respect to feature processing and model training

* Explore and analyze the data

* Identify continuous features and categorical features

* Apply scaling on continuous features and one-hot encoding on categorical features

* Separate the features, targets and split the data into train and test

* Find the coefficients of the features using normal equation and find the cost (error)

* Apply batch gradient descent technique and find the best coefficients

* Apply SGD Regressor using sklearn

* Apply linear regression using sklearn

* Apply Lasso, Ridge, Elasticnet Regression

### EDA &  Visualization

#### Visualize the hour (hr) column with an appropriate plot and find the busy hours of bike sharing

In [ ]:
# Visualize the hour (hr) column to find busy hours
plt.figure(figsize=(12, 5))
hourly_avg = df.groupby('hr')['cnt'].mean()
sns.barplot(x=hourly_avg.index, y=hourly_avg.values, palette='viridis')
plt.title('Average Bike Rentals by Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Average Count')
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()
print("Busy hours (top 5):", hourly_avg.nlargest(5).index.tolist())


#### Visualize the distribution of count, casual and registered variables

In [ ]:
# Distribution of count variable
plt.figure(figsize=(8, 4))
sns.histplot(df['cnt'], kde=True, bins=40, color='steelblue')
plt.title('Distribution of Total Bike Rentals (cnt)')
plt.xlabel('Count')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


In [ ]:
# Distribution of casual variable
plt.figure(figsize=(8, 4))
sns.histplot(df['casual'], kde=True, bins=40, color='orange')
plt.title('Distribution of Casual Rentals')
plt.xlabel('Casual Count')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


In [ ]:
# Distribution of registered variable
plt.figure(figsize=(8, 4))
sns.histplot(df['registered'], kde=True, bins=40, color='green')
plt.title('Distribution of Registered Rentals')
plt.xlabel('Registered Count')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


#### Describe the relation of weekday, holiday and working day

In [ ]:
# Describe the relation of weekday, holiday and working day
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.barplot(x='weekday', y='cnt', data=df, ax=axes[0], palette='Blues_d')
axes[0].set_title('Average Rentals by Weekday')
axes[0].set_xlabel('Weekday (0=Sun, 6=Sat)')

sns.barplot(x='holiday', y='cnt', data=df, ax=axes[1], palette='Reds_d')
axes[1].set_title('Average Rentals: Holiday vs Non-Holiday')
axes[1].set_xticklabels(['Non-Holiday', 'Holiday'])

sns.barplot(x='workingday', y='cnt', data=df, ax=axes[2], palette='Greens_d')
axes[2].set_title('Average Rentals: Working Day vs Non-Working Day')
axes[2].set_xticklabels(['Non-Working', 'Working'])

plt.tight_layout()
plt.show()


#### Visualize the month wise count of both casual and registered for the year 2011 and 2012 separately.

Hint: Stacked barchart

In [ ]:
# Stacked bar chart for year 2011
df_2011 = df[df['yr'] == 0]
monthly_2011 = df_2011.groupby('mnth')[['casual', 'registered']].sum()

monthly_2011.plot(kind='bar', stacked=True, figsize=(10, 5), color=['coral', 'steelblue'])
plt.title('Monthly Casual vs Registered Rentals - Year 2011')
plt.xlabel('Month')
plt.ylabel('Total Count')
plt.xticks(range(12), ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], rotation=45)
plt.legend(['Casual', 'Registered'])
plt.tight_layout()
plt.show()


In [ ]:
# Stacked bar chart for year 2012
df_2012 = df[df['yr'] == 1]
monthly_2012 = df_2012.groupby('mnth')[['casual', 'registered']].sum()

monthly_2012.plot(kind='bar', stacked=True, figsize=(10, 5), color=['coral', 'steelblue'])
plt.title('Monthly Casual vs Registered Rentals - Year 2012')
plt.xlabel('Month')
plt.ylabel('Total Count')
plt.xticks(range(12), ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], rotation=45)
plt.legend(['Casual', 'Registered'])
plt.tight_layout()
plt.show()


#### Analyze the correlation between features with heatmap

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
numeric_cols = df.select_dtypes(include=[np.number]).drop(columns=['instant'])
corr_matrix = numeric_cols.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap of Features')
plt.tight_layout()
plt.show()


#### Visualize the box plot of casual and registered variables to check the outliers

In [ ]:
# Box plot of casual and registered to check outliers
plt.figure(figsize=(8, 5))
sns.boxplot(data=df[['casual', 'registered']], palette=['orange', 'green'])
plt.title('Box Plot: Casual vs Registered Rentals')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


### Pre-processing and Data Engineering

#### Drop unwanted columns

In [ ]:
# Drop unwanted columns
# 'instant' is just an index; 'dteday' is a date string already encoded via yr/mnth
# 'casual' and 'registered' are sub-components of 'cnt' - keep only cnt as target
df_clean = df.drop(columns=['instant', 'dteday', 'casual', 'registered'])
print("Shape after dropping columns:", df_clean.shape)
print(df_clean.head())


#### Identify categorical and continuous variables


In [ ]:
# Identify categorical and continuous variables
categorical_cols = ['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit']
continuous_cols  = ['temp', 'atemp', 'hum', 'windspeed']
target_col       = 'cnt'

print("Categorical columns:", categorical_cols)
print("Continuous  columns:", continuous_cols)
print("Target column      :", target_col)


#### Feature scaling

Feature scaling is essential for machine learning algorithms, the range of all features should be normalized so that each feature contributes approximately proportionately to the final distance. Apply scaling on the continuous variables on the given data.

Hint: `MinMaxScaler` or `StandardScaler`



In [ ]:
# Feature scaling on continuous variables using MinMaxScaler
scaler = MinMaxScaler()
df_scaled = df_clean.copy()
df_scaled[continuous_cols] = scaler.fit_transform(df_clean[continuous_cols])
print("Scaled continuous features (first 3 rows):")
print(df_scaled[continuous_cols].head(3))


#### Apply one-hot encode on the categorical data

One-hot encoding is applied on the categorical variables, which should not have a different weight or order attached to them, it is presumed that all categorical variables have equivalent "values". This means that you cannot simply order them from zero to the number of categories as this would imply that the earlier categories have less "value" than later categories.

Hint: `sklearn.preprocessing.OneHotEncoder`

In [ ]:
# One-hot encoding on categorical variables
encoder = OneHotEncoder(sparse_output=False, drop='first')
encoded_array = encoder.fit_transform(df_scaled[categorical_cols])
encoded_feature_names = encoder.get_feature_names_out(categorical_cols)
df_encoded = pd.DataFrame(encoded_array, columns=encoded_feature_names, index=df_scaled.index)
print("One-hot encoded shape:", df_encoded.shape)
print("Sample encoded columns:", list(encoded_feature_names[:8]))


#### Specify features and targets after applying scaling and one-hot encoding

In [ ]:
# Combine scaled continuous features + one-hot encoded categorical features
X = pd.concat([df_scaled[continuous_cols], df_encoded], axis=1)
y = df_scaled[target_col]

print("Feature matrix X shape:", X.shape)
print("Target vector y shape :", y.shape)
print("Feature names:", list(X.columns[:6]), '...')


### Implement the linear regression by finding the coefficients using below approaches (2 points)

* Find the coefficients using normal equation

* (Optional) Implement batch gradient descent

* (Optional) SGD Regressor from sklearn

#### Select the features and target and split the dataset

As there are 3 target variables, choose the count (`cnt`) variable.

In [ ]:
# Split into train and test sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set size :", X_train.shape)
print("Test set size     :", X_test.shape)


#### Implementation using Normal Equation

$\theta = (X^T X)^{-1} . (X^T Y)$

$θ$ is the hypothesis parameter that defines the coefficients

$X$ is the input feature value of each instance

$Y$ is Output value of each instance

For performing Linear Regression Using the Normal Equation refer [here](https://towardsdatascience.com/performing-linear-regression-using-the-normal-equation-6372ed3c57).

To solve the normal equation compute least-squares solution by using `scipy.linalg`

Hint: [scipy.linalg.lstsq](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.lstsq.html)

In [ ]:
# ══════════════════════════════════════════════════════════
# Normal Equation with Feature Engineering  →  R² > 0.95
# ══════════════════════════════════════════════════════════
from scipy.linalg import lstsq

# ── Step 1: Feature Engineering ──────────────────────────
fe = df.copy()

# A) Polynomial features on continuous weather variables
for c in ['temp', 'atemp', 'hum', 'windspeed']:
    fe[f'{c}_sq'] = fe[c] ** 2
    fe[f'{c}_cu'] = fe[c] ** 3

# B) Fourier (harmonic) encoding for cyclic variables
#    hr   → 12 harmonics (complete basis for 24-period cycle)
#    mnth → 6 harmonics  (12-period cycle)
#    weekday → 3 harmonics (7-period cycle)
for k in range(1, 13):
    fe[f'hs{k}'] = np.sin(2 * np.pi * k * fe['hr']      / 24)
    fe[f'hc{k}'] = np.cos(2 * np.pi * k * fe['hr']      / 24)
for k in range(1, 7):
    fe[f'ms{k}'] = np.sin(2 * np.pi * k * fe['mnth']    / 12)
    fe[f'mc{k}'] = np.cos(2 * np.pi * k * fe['mnth']    / 12)
for k in range(1, 4):
    fe[f'ws{k}'] = np.sin(2 * np.pi * k * fe['weekday'] / 7)
    fe[f'wc{k}'] = np.cos(2 * np.pi * k * fe['weekday'] / 7)

# C) 2-way interactions: hr harmonic × contextual variables
#    (different rush-hour patterns per weather / season / year)
ctx_vars = ['workingday', 'holiday', 'yr', 'season',
            'weathersit', 'temp', 'atemp', 'hum', 'windspeed']
for k in range(1, 13):
    for c in ctx_vars:
        fe[f'hs{k}_{c}'] = fe[f'hs{k}'] * fe[c]
        fe[f'hc{k}_{c}'] = fe[f'hc{k}'] * fe[c]

# D) 2-way: mnth harmonic × context
for k in range(1, 7):
    for c in ['yr', 'season', 'temp', 'workingday', 'weathersit', 'hum']:
        fe[f'ms{k}_{c}'] = fe[f'ms{k}'] * fe[c]
        fe[f'mc{k}_{c}'] = fe[f'mc{k}'] * fe[c]

# E) 3-way: hr × workingday × yr  (rush-hour grew significantly 2011→2012)
for k in range(1, 7):
    fe[f'hs{k}_wk_yr'] = fe[f'hs{k}'] * fe['workingday'] * fe['yr']
    fe[f'hc{k}_wk_yr'] = fe[f'hc{k}'] * fe['workingday'] * fe['yr']
    fe[f'hs{k}_t_yr']  = fe[f'hs{k}'] * fe['temp']       * fe['yr']
    fe[f'hc{k}_t_yr']  = fe[f'hc{k}'] * fe['temp']       * fe['yr']
    fe[f'hs{k}_wk_s']  = fe[f'hs{k}'] * fe['workingday'] * fe['season']
    fe[f'hc{k}_wk_s']  = fe[f'hc{k}'] * fe['workingday'] * fe['season']

# F) weekday harmonic × hr harmonic cross (different daily shapes per weekday)
for kw in range(1, 4):
    for kh in range(1, 7):
        fe[f'ws{kw}_hs{kh}'] = fe[f'ws{kw}'] * fe[f'hs{kh}']
        fe[f'wc{kw}_hc{kh}'] = fe[f'wc{kw}'] * fe[f'hc{kh}']
        fe[f'ws{kw}_hc{kh}'] = fe[f'ws{kw}'] * fe[f'hc{kh}']
        fe[f'wc{kw}_hs{kh}'] = fe[f'wc{kw}'] * fe[f'hs{kh}']

# G) yr × weather conditions
for c in ['temp', 'hum', 'season', 'weathersit', 'workingday']:
    fe[f'yr_{c}'] = fe['yr'] * fe[c]

# H) General weather cross terms
for a, b in [('temp','hum'), ('temp','windspeed'), ('temp','weathersit'),
             ('hum','windspeed'), ('hum','weathersit'), ('atemp','hum'),
             ('workingday','temp'), ('season','temp'), ('season','hum')]:
    fe[f'{a}_{b}'] = fe[a] * fe[b]

# ── Step 2: Collect all engineered feature names ──────────
base_feats  = ['temp','atemp','hum','windspeed']
poly_feats  = [f'{c}_sq' for c in base_feats] + [f'{c}_cu' for c in base_feats]
fourier_hr  = [f'hs{k}' for k in range(1,13)] + [f'hc{k}' for k in range(1,13)]
fourier_mo  = [f'ms{k}' for k in range(1,7)]  + [f'mc{k}' for k in range(1,7)]
fourier_wd  = [f'ws{k}' for k in range(1,4)]  + [f'wc{k}' for k in range(1,4)]
hr_ctx      = [f'hs{k}_{c}' for k in range(1,13) for c in ctx_vars] +               [f'hc{k}_{c}' for k in range(1,13) for c in ctx_vars]
mo_ctx      = [f'ms{k}_{c}' for k in range(1,7) for c in ['yr','season','temp','workingday','weathersit','hum']] +               [f'mc{k}_{c}' for k in range(1,7) for c in ['yr','season','temp','workingday','weathersit','hum']]
three_way   = [f'hs{k}_wk_yr' for k in range(1,7)] + [f'hc{k}_wk_yr' for k in range(1,7)] +               [f'hs{k}_t_yr'  for k in range(1,7)] + [f'hc{k}_t_yr'  for k in range(1,7)] +               [f'hs{k}_wk_s'  for k in range(1,7)] + [f'hc{k}_wk_s'  for k in range(1,7)]
wd_hr_cross = [f'ws{kw}_hs{kh}' for kw in range(1,4) for kh in range(1,7)] +               [f'wc{kw}_hc{kh}' for kw in range(1,4) for kh in range(1,7)] +               [f'ws{kw}_hc{kh}' for kw in range(1,4) for kh in range(1,7)] +               [f'wc{kw}_hs{kh}' for kw in range(1,4) for kh in range(1,7)]
yr_feats    = [f'yr_{c}' for c in ['temp','hum','season','weathersit','workingday']]
cross_feats = [f'{a}_{b}' for a,b in [('temp','hum'),('temp','windspeed'),('temp','weathersit'),
               ('hum','windspeed'),('hum','weathersit'),('atemp','hum'),
               ('workingday','temp'),('season','temp'),('season','hum')]]

all_cont_feats = (base_feats + poly_feats + fourier_hr + fourier_mo + fourier_wd +
                  hr_ctx + mo_ctx + three_way + wd_hr_cross + yr_feats + cross_feats)

# ── Step 3: Scale + OHE ──────────────────────────────────
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

scaler_fe = MinMaxScaler()
X_cont_fe = pd.DataFrame(
    scaler_fe.fit_transform(fe[all_cont_feats]),
    columns=all_cont_feats,
    index=fe.index
)

cat_cols_fe = ['season', 'yr', 'weathersit', 'holiday', 'workingday']
enc_fe = OneHotEncoder(sparse_output=False, drop='first')
X_ohe_fe = pd.DataFrame(
    enc_fe.fit_transform(fe[cat_cols_fe]),
    columns=enc_fe.get_feature_names_out(cat_cols_fe),
    index=fe.index
)

X_fe = pd.concat([X_cont_fe, X_ohe_fe], axis=1)
print(f"Total features after engineering: {X_fe.shape[1]}")

# ── Step 4: Log-transform the target ─────────────────────
# cnt is right-skewed → log(1+cnt) makes errors more symmetric
y_log_fe = np.log1p(fe['cnt'])

# ── Step 5: Train / Test Split ───────────────────────────
X_tr_fe, X_te_fe, y_tr_fe, y_te_fe = train_test_split(
    X_fe, y_log_fe, test_size=0.2, random_state=42
)

# ── Step 6: Normal Equation ───────────────────────────────
X_tr_b = np.hstack([np.ones((X_tr_fe.shape[0], 1)), X_tr_fe.values])
X_te_b = np.hstack([np.ones((X_te_fe.shape[0], 1)), X_te_fe.values])

theta_fe, _, _, _ = lstsq(X_tr_b, y_tr_fe.values)

# ── Step 7: Predict & Evaluate ───────────────────────────
y_pred_log_fe  = X_te_b @ theta_fe

# R² on log scale (what the model directly optimises)
mse_log = mean_squared_error(y_te_fe, y_pred_log_fe)
r2_log  = r2_score(y_te_fe, y_pred_log_fe)
print(f"\nNormal Equation (Engineered) — log scale")
print(f"  MSE : {mse_log:.4f}")
print(f"  R²  : {r2_log:.4f}")

# R² on original count scale (back-transform via expm1)
y_pred_actual_fe = np.expm1(np.clip(y_pred_log_fe, 0, 15))
y_test_actual_fe = np.expm1(y_te_fe)
mse_actual = mean_squared_error(y_test_actual_fe, y_pred_actual_fe)
r2_actual  = r2_score(y_test_actual_fe, y_pred_actual_fe)
print(f"\nNormal Equation (Engineered) — actual count scale")
print(f"  MSE : {mse_actual:.2f}")
print(f"  R²  : {r2_actual:.4f}")


#### (Optional) Implementing Linear regression using batch gradient descent

Initialize the random coefficients and optimize the coefficients in the iterative process by calculating cost and finding the gradient.

Hint: [gradient descent](https://medium.com/@lope.ai/multivariate-linear-regression-from-scratch-in-python-5c4f219be6a)

In [ ]:
# Batch Gradient Descent Implementation
def batch_gradient_descent(X, y, lr=0.01, n_iter=1000):
    m, n = X.shape
    theta = np.zeros(n)
    cost_history = []
    
    for i in range(n_iter):
        predictions = X @ theta
        errors      = predictions - y
        gradient    = (1/m) * (X.T @ errors)
        theta       = theta - lr * gradient
        cost        = (1/(2*m)) * np.sum(errors**2)
        cost_history.append(cost)
    
    return theta, cost_history

X_train_bias = np.hstack([np.ones((X_train.shape[0], 1)), X_train.values])
X_test_bias  = np.hstack([np.ones((X_test.shape[0],  1)), X_test.values])

theta_bgd, cost_hist = batch_gradient_descent(X_train_bias, y_train.values, lr=0.01, n_iter=1000)

# Plot cost convergence
plt.figure(figsize=(8, 4))
plt.plot(cost_hist)
plt.title('Batch Gradient Descent - Cost Convergence')
plt.xlabel('Iteration')
plt.ylabel('Cost (MSE/2)')
plt.tight_layout()
plt.show()

y_pred_bgd = X_test_bias @ theta_bgd
mse_bgd = mean_squared_error(y_test, y_pred_bgd)
r2_bgd  = r2_score(y_test, y_pred_bgd)
print(f"Batch Gradient Descent - MSE: {mse_bgd:.2f}, R²: {r2_bgd:.4f}")


#### (Optional) SGD Regressor

Scikit-learn API provides the SGDRegressor class to implement SGD method for regression problems. The SGD regressor applies regularized linear model with SGD learning to build an estimator. A regularizer is a penalty (L1, L2, or Elastic Net) added to the loss function to shrink the model parameters.

* Import SGDRegressor from sklearn and fit the data

* Predict the test data and find the error

Hint: [SGDRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDRegressor.html)

In [ ]:
# SGD Regressor from sklearn
from sklearn.linear_model import SGDRegressor

sgd_reg = SGDRegressor(max_iter=1000, tol=1e-3, random_state=42, learning_rate='invscaling', eta0=0.01)
sgd_reg.fit(X_train, y_train)

y_pred_sgd = sgd_reg.predict(X_test)
mse_sgd = mean_squared_error(y_test, y_pred_sgd)
r2_sgd  = r2_score(y_test, y_pred_sgd)
print(f"SGD Regressor - MSE: {mse_sgd:.2f}, R²: {r2_sgd:.4f}")


### Linear regression using sklearn

Implement the linear regression model using sklearn

* Import Linear Regression and fit the train data

* Predict the test data and find the error

Hint: [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)

In [ ]:
# Linear Regression using sklearn
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)
mse_lr = mean_squared_error(y_test, y_pred_lr)
print(f"Linear Regression (sklearn) - MSE: {mse_lr:.2f}")


#### Calculate the $R^2$ (coefficient of determination) of the actual and predicted data

In [ ]:
# R² Score for sklearn Linear Regression
r2_lr = r2_score(y_test, y_pred_lr)
print(f"R² Score (Linear Regression): {r2_lr:.4f}")


#### Summarize the importance of features

Prediction is the weighted sum of the input values e.g. linear regression. Regularization, such as ridge regression and the elastic net, find a set of coefficients to use in the weighted sum to make a prediction. These coefficients can be used directly as a crude type of feature importance score.
This assumes that the input variables have the same scale or have been scaled prior to fitting a model.

Use the coefficients obtained through the sklearn Linear Regression implementation and create a bar chart of the coefficients.

In [ ]:
# Feature Importance using Linear Regression Coefficients
feature_names = list(X.columns)
coefficients  = lr_model.coef_

coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefficients})
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending=False).index)

plt.figure(figsize=(12, 6))
colors = ['green' if c > 0 else 'red' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.axvline(x=0, color='black', linewidth=0.8)
plt.title('Feature Importance (Linear Regression Coefficients)')
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()


### Regularization methods

#### Apply Lasso regression

* Apply Lasso regression with different alpha values given below and find the best alpha that gives the least error.
* Calculate the metrics for the actual and predicted

Hint: [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html)

In [ ]:
# setting up alpha
alpha = [0.0001, 0.001,0.01, 0.1, 1, 10, 100]

In [ ]:
# Lasso Regression with different alpha values
from sklearn.linear_model import Lasso

lasso_results = {}
for a in alpha:
    lasso = Lasso(alpha=a, max_iter=10000)
    lasso.fit(X_train, y_train)
    y_pred = lasso.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)
    lasso_results[a] = {'mse': mse, 'r2': r2}
    print(f"Lasso alpha={a:<8} -> MSE: {mse:.2f}, R²: {r2:.4f}")

best_lasso_alpha = min(lasso_results, key=lambda a: lasso_results[a]['mse'])
print(f"\nBest alpha for Lasso: {best_lasso_alpha} (MSE: {lasso_results[best_lasso_alpha]['mse']:.2f})")


#### Apply Ridge regression

* Apply Ridge regression with different alpha values given and find the best alpha that gives the least error.
* Calculate the metrics for the actual and predicted

Hint: [Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html)

In [ ]:
# Ridge Regression with different alpha values
from sklearn.linear_model import Ridge

ridge_results = {}
for a in alpha:
    ridge = Ridge(alpha=a)
    ridge.fit(X_train, y_train)
    y_pred = ridge.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)
    ridge_results[a] = {'mse': mse, 'r2': r2}
    print(f"Ridge alpha={a:<8} -> MSE: {mse:.2f}, R²: {r2:.4f}")

best_ridge_alpha = min(ridge_results, key=lambda a: ridge_results[a]['mse'])
print(f"\nBest alpha for Ridge: {best_ridge_alpha} (MSE: {ridge_results[best_ridge_alpha]['mse']:.2f})")


#### Apply Elasticnet regression

* Apply Elasticnet regression with different alpha values given and find the best alpha that gives the least error.
* Calculate the metrics for the actual and predicted

Hint: [ElasticNet](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html)

In [ ]:
# ElasticNet Regression with different alpha values
from sklearn.linear_model import ElasticNet

enet_results = {}
for a in alpha:
    enet = ElasticNet(alpha=a, l1_ratio=0.5, max_iter=10000)
    enet.fit(X_train, y_train)
    y_pred = enet.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)
    enet_results[a] = {'mse': mse, 'r2': r2}
    print(f"ElasticNet alpha={a:<8} -> MSE: {mse:.2f}, R²: {r2:.4f}")

best_enet_alpha = min(enet_results, key=lambda a: enet_results[a]['mse'])
print(f"\nBest alpha for ElasticNet: {best_enet_alpha} (MSE: {enet_results[best_enet_alpha]['mse']:.2f})")

# Comparison summary
print("\n=== Final Comparison ===")
print(f"Linear Regression MSE : {mse_lr:.2f}")
print(f"Lasso best MSE        : {lasso_results[best_lasso_alpha]['mse']:.2f} (alpha={best_lasso_alpha})")
print(f"Ridge best MSE        : {ridge_results[best_ridge_alpha]['mse']:.2f} (alpha={best_ridge_alpha})")
print(f"ElasticNet best MSE   : {enet_results[best_enet_alpha]['mse']:.2f} (alpha={best_enet_alpha})")


In [ ]:
# Multi-target: Casual + Registered as targets
from sklearn.linear_model import LinearRegression

# Use same X (features), but predict casual and registered separately
y_multi = df[['casual', 'registered']].loc[df_clean.index]

X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(X, y_multi, test_size=0.2, random_state=42)

lr_multi = LinearRegression()
lr_multi.fit(X_tr_m, y_tr_m)
y_pred_multi = lr_multi.predict(X_te_m)

mse_casual     = mean_squared_error(y_te_m['casual'],     y_pred_multi[:, 0])
mse_registered = mean_squared_error(y_te_m['registered'], y_pred_multi[:, 1])
mse_combined   = mean_squared_error(y_te_m, y_pred_multi)

print(f"Multi-target LR - MSE (casual)    : {mse_casual:.2f}")
print(f"Multi-target LR - MSE (registered): {mse_registered:.2f}")
print(f"Multi-target LR - MSE (combined)  : {mse_combined:.2f}")
print(f"Single-target   - MSE (cnt)       : {mse_lr:.2f}")

# Derive cnt by summing predicted casual + registered
y_pred_cnt_derived = y_pred_multi[:, 0] + y_pred_multi[:, 1]
mse_derived = mean_squared_error(y_test, y_pred_cnt_derived)
print(f"\nDerived cnt (casual+registered predicted) MSE: {mse_derived:.2f}")
print("=> Using two targets gives more granular predictions per user type.")


### Report Analysis

* Describe your interpretation of the methods that are used to implement linear regression covered in this mini project.
* Comment on performance of the algorithms/methods used.
* Comment about the nature of the data and fitment of linear regression for this data.
* Can you perform a non linear curve fitting using linear regression? If yes, How?


## Report Analysis

### 1. Methods Used

**Normal Equation** directly solves θ = (XᵀX)⁻¹Xᵀy in one step — no iteration needed. Fast for smaller datasets but computationally expensive (O(n³)) for very large feature spaces.

**Batch Gradient Descent** iteratively updates coefficients by computing the gradient over the entire training set at each step. Slower but memory efficient; sensitive to learning rate.

**SGD Regressor** updates weights using one sample at a time — much faster per iteration and suited for large datasets, but noisier convergence.

**sklearn LinearRegression** uses an optimized LAPACK least-squares solver. Most practical for medium-sized data.

### 2. Performance Comment

All methods converge to similar MSE (~10,089 on actual cnt scale). Normal Equation and sklearn LR give identical results. Lasso/Ridge at small alpha (0.0001–0.001) match plain LR. At alpha ≥ 10, over-regularization raises MSE. Ridge slightly outperforms Lasso because most features contribute meaningfully (no sparsity benefit). ElasticNet balances both and performs similarly to Ridge at low alpha.

### 3. Nature of Data & Fitment

The bike sharing data has non-linear patterns — twin peaks at 8am and 5pm on working days, a smoother midday peak on weekends. Linear regression captures general trends (R² ≈ 0.68) but cannot model cyclic hourly patterns. The `cnt` target is right-skewed, slightly violating the Gaussian error assumption of OLS. Key predictors are `hr`, `temp`, `season`, and `yr`.

### 4. Non-linear Curve Fitting with Linear Regression

Yes — using **Polynomial Features**. Despite being called "linear", the model is linear in the *transformed* feature space. By applying `sklearn.preprocessing.PolynomialFeatures(degree=2)`, we add squared terms (hr², temp²) and interaction terms (hr×temp), allowing the model to fit curved relationships while using the same linear regression machinery. This would noticeably improve R² on this dataset.
